# Imports

In [35]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load data

In [36]:
project_root = Path.cwd().parent
raw_path = project_root / "data" / "spy_raw.csv"

df = pd.read_csv(raw_path, header=[0, 1], index_col=0)
df.columns = df.columns.get_level_values(0)
df.index.name = "Date"
df = df.reset_index()

df.head()

Price,Date,Close,High,Low,Open,Volume
0,2010-01-04,85.027939,85.072953,83.662450,84.307682,118944600
1,2010-01-05,85.253006,85.290522,84.667797,84.975410,111579900
2,2010-01-06,85.313065,85.523139,85.102990,85.170512,116074400
3,2010-01-07,85.673187,85.778224,84.915414,85.155500,131091100
4,2010-01-08,85.958275,85.995791,85.275533,85.448092,126402800


# Inspect data

In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4073 entries, 0 to 4072
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    4073 non-null   str    
 1   Close   4073 non-null   float64
 2   High    4073 non-null   float64
 3   Low     4073 non-null   float64
 4   Open    4073 non-null   float64
 5   Volume  4073 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 191.1 KB


In [38]:
df.isna().sum()

Price
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [39]:
df.describe()

Price,Close,High,Low,Open,Volume
count,4073.000000,4073.000000,4073.000000,4073.000000,4.073000e+03
mean,273.827044,275.243630,272.174265,273.775705,1.090606e+08
std,161.630388,162.423753,160.686150,161.603937,6.763739e+07
min,77.359520,78.282991,76.549604,78.048340,2.027000e+07
25%,148.880173,149.507290,148.293708,148.920887,6.537080e+07
50%,232.708664,234.675908,230.245372,232.274527,8.925780e+07
75%,391.321533,393.538675,389.187512,390.995903,1.322564e+08
max,695.489990,697.840027,693.940002,697.049988,7.178287e+08


# Convert date and sort

In [40]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.head()

Price,Date,Close,High,Low,Open,Volume
0,2010-01-04,85.027939,85.072953,83.662450,84.307682,118944600
1,2010-01-05,85.253006,85.290522,84.667797,84.975410,111579900
2,2010-01-06,85.313065,85.523139,85.102990,85.170512,116074400
3,2010-01-07,85.673187,85.778224,84.915414,85.155500,131091100
4,2010-01-08,85.958275,85.995791,85.275533,85.448092,126402800


# Create target

In [41]:
df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df[["Date","Open",  "Close", "target"]].head(10)

Price,Date,Open,Close,target
0,2010-01-04,84.307682,85.027939,1
1,2010-01-05,84.975410,85.253006,1
2,2010-01-06,85.170512,85.313065,1
3,2010-01-07,85.155500,85.673187,1
4,2010-01-08,85.448092,85.958275,1
5,2010-01-11,86.340932,86.078339,0
6,2010-01-12,85.508117,85.275536,1
7,2010-01-13,85.493097,85.995781,1
8,2010-01-14,85.898251,86.228371,0
9,2010-01-15,86.078355,85.260559,1


# Create features

In [42]:
df["return_1d"] = df["Close"].pct_change()
df["return_5d"] = df["Close"].pct_change(5)

df["ma_5"] = df["Close"].rolling(5).mean()
df["ma_10"] = df["Close"].rolling(10).mean()
df["ma_20"] = df["Close"].rolling(20).mean()

df["ma_5_rel"] = df["Close"] / df["ma_5"] - 1
df["ma_10_rel"] = df["Close"] / df["ma_10"] - 1
df["ma_20_rel"] = df["Close"] / df["ma_20"] - 1

df["mom_2d"] = df["Close"].pct_change(2)
df["mom_10d"] = df["Close"].pct_change(10)
df["mom_20d"] = df["Close"].pct_change(20)

df["vol_5"] = df["return_1d"].rolling(5).std()
df["vol_20"] = df["return_1d"].rolling(20).std()
df["vol_ratio"] = df["vol_5"] / df["vol_20"]

df["volume_change_1d"] = df["Volume"].pct_change()

df["high_10"] = df["High"].rolling(10).max()
df["low_10"] = df["Low"].rolling(10).min()
df["range_pos_10"] = (df["Close"] - df["low_10"]) / (df["high_10"] - df["low_10"])

df["high_20"] = df["High"].rolling(20).max()
df["low_20"] = df["Low"].rolling(20).min()
df["range_pos_20"] = (df["Close"] - df["low_20"]) / (df["high_20"] - df["low_20"])

df["ma_spread_5_20"] = df["ma_5"] / df["ma_20"] - 1
df["ma_spread_10_20"] = df["ma_10"] / df["ma_20"] - 1

df["intraday_return"] = df["Close"] / df["Open"] - 1
df["high_low_range"] = df["High"] / df["Low"] - 1
df["close_to_high"] = df["Close"] / df["High"] - 1
df["close_to_low"] = df["Close"] / df["Low"] - 1

df = df.drop(columns=["ma_5", "ma_10", "ma_20", "high_10", "low_10", "high_20", "low_20"])

df.head()

Price,Date,Close,High,Low,Open,Volume,target,return_1d,return_5d,ma_5_rel,...,vol_ratio,volume_change_1d,range_pos_10,range_pos_20,ma_spread_5_20,ma_spread_10_20,intraday_return,high_low_range,close_to_high,close_to_low
0,2010-01-04,85.027939,85.072953,83.662450,84.307682,118944600,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.008543,0.016859,-0.000529,0.016321
1,2010-01-05,85.253006,85.290522,84.667797,84.975410,111579900,1,0.002647,NaN,NaN,...,NaN,-0.061917,NaN,NaN,NaN,NaN,0.003267,0.007355,-0.000440,0.006912
2,2010-01-06,85.313065,85.523139,85.102990,85.170512,116074400,1,0.000704,NaN,NaN,...,NaN,0.040281,NaN,NaN,NaN,NaN,0.001674,0.004937,-0.002456,0.002468
3,2010-01-07,85.673187,85.778224,84.915414,85.155500,131091100,1,0.004221,NaN,NaN,...,NaN,0.129371,NaN,NaN,NaN,NaN,0.006079,0.010161,-0.001225,0.008924
4,2010-01-08,85.958275,85.995791,85.275533,85.448092,126402800,1,0.003328,NaN,0.006006,...,NaN,-0.035764,NaN,NaN,NaN,NaN,0.005971,0.008446,-0.000436,0.008006


# Inspect engineered features

In [43]:
df.isna().sum()

Price
Date                 0
Close                0
High                 0
Low                  0
Open                 0
Volume               0
target               0
return_1d            1
return_5d            5
ma_5_rel             4
ma_10_rel            9
ma_20_rel           19
mom_2d               2
mom_10d             10
mom_20d             20
vol_5                5
vol_20              20
vol_ratio           20
volume_change_1d     1
range_pos_10         9
range_pos_20        19
ma_spread_5_20      19
ma_spread_10_20     19
intraday_return      0
high_low_range       0
close_to_high        0
close_to_low         0
dtype: int64

# Drop missing values

In [44]:
df = df.dropna().reset_index(drop=True)

df.head()

Price,Date,Close,High,Low,Open,Volume,target,return_1d,return_5d,ma_5_rel,...,vol_ratio,volume_change_1d,range_pos_10,range_pos_20,ma_spread_5_20,ma_spread_10_20,intraday_return,high_low_range,close_to_high,close_to_low
0,2010-02-02,82.814667,82.972223,81.689264,81.974369,216327900,0,0.012103,0.009789,0.012233,...,1.195418,0.151507,0.437069,0.398994,-0.027178,-0.019462,0.010251,0.015705,-0.001899,0.013777
1,2010-02-03,82.402031,82.889707,82.161945,82.439541,172730700,0,-0.004983,0.000000,0.007190,...,1.217450,-0.201533,0.370218,0.329552,-0.025527,-0.021425,-0.000455,0.008858,-0.005883,0.002922
2,2010-02-04,79.858604,81.801798,79.843596,81.764288,356715700,1,-0.030866,-0.019618,-0.020070,...,1.514461,1.065155,0.003760,0.002294,-0.026170,-0.022952,-0.023307,0.024525,-0.023755,0.000188
3,2010-02-05,80.023659,80.188713,78.463099,79.948627,493585800,0,0.002067,-0.006797,-0.016723,...,1.495205,0.383695,0.346089,0.196970,-0.024185,-0.021936,0.000938,0.021993,-0.002058,0.019889
4,2010-02-08,79.445938,80.526327,79.385915,80.083665,224166900,1,-0.007219,-0.029067,-0.018083,...,1.297265,-0.545840,0.217967,0.124052,-0.026086,-0.021607,-0.007963,0.014365,-0.013417,0.000756


In [45]:
df.shape

(4053, 27)

# Save final dataset

In [46]:
features_path = project_root / "data" / "spy_features.csv"
df.to_csv(features_path, index=False)

print(features_path)

C:\Users\999ch\Documents\Quant Projects\spy-direction-prediction\data\spy_features.csv
